In [1]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import vgg19
from skimage.color import lab2rgb
from tqdm import tqdm

#noinspection PyUnresolvedReferences
from model_resnet_unet_lab import ResNetUNet
#noinspection PyUnresolvedReferences
from dataset_lab import get_dataloader

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

Устройство: cpu


In [ ]:
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 2e-4
DATA_DIR = '../data/processed/test'

In [ ]:
class VisualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(weights='IMAGENET1K_V1').features
        self.slice = nn.Sequential(*list(vgg.children())[:18]).eval()
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, fake_ab, real_ab, l_batch):
        fake_img = torch.cat([l_batch, fake_ab], dim=1)
        real_img = torch.cat([l_batch, real_ab], dim=1)
        return nn.functional.l1_loss(self.slice(fake_img), self.slice(real_img))

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to C:\Users\Denis/.cache\torch\hub\checkpoints\vgg19-dcbb9e9d.pth


  2%|▏         | 11.4M/548M [01:00<1:16:27, 123kB/s]

In [ ]:
criterion_l1 = nn.L1Loss()
criterion_perceptual = VisualLoss().to(device)

model = ResNetUNet(n_class=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
def lab_to_rgb(l_tensor, ab_tensor):
    """Преобразование тензоров обратно в RGB для matplotlib"""
    l = (l_tensor.detach().cpu().numpy() + 1.0) * 50.0
    ab = ab_tensor.detach().cpu().numpy() * 128.0
    lab = np.concatenate([l, ab], axis=0).transpose(1, 2, 0)
    rgb = lab2rgb(lab.astype(np.float64))
    return rgb

def show_prediction(model, l_batch, ab_batch, epoch):
    model.eval()
    with torch.no_grad():
        pred_ab = model(l_batch[:1].to(device))

    l_img = l_batch[0]
    real_ab = ab_batch[0]
    pred_ab = pred_ab[0]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(l_img[0].cpu(), cmap='gray')
    plt.title("Вход (L)")

    plt.subplot(1, 3, 2)
    plt.imshow(lab_to_rgb(l_img, pred_ab))
    plt.title(f"Результат (Эпоха {epoch})")

    plt.subplot(1, 3, 3)
    plt.imshow(lab_to_rgb(l_img, real_ab))
    plt.title("Оригинал")
    plt.show()

In [ ]:
model = ResNetUNet(n_class=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
train_loader = get_dataloader(DATA_DIR, batch_size=BATCH_SIZE)

# Списки для хранения истории лоссов (для графиков)
history = {
    'total_loss': [],
    'l1_loss': [],
    'vgg_loss': []
}

print("Начинаем обучение...")
for epoch in range(EPOCHS):
    model.train()
    running_total = 0.0
    running_l1 = 0.0
    running_vgg = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for l_batch, ab_batch in loop:
        l_batch, ab_batch = l_batch.to(device), ab_batch.to(device)

        optimizer.zero_grad()

        # Forward pass
        pred_ab = model(l_batch)

        # Вычисление лоссов
        loss_l1 = criterion_l1(pred_ab, ab_batch)
        loss_vgg = criterion_perceptual(pred_ab, ab_batch, l_batch)

        total_loss = loss_l1 + 0.1 * loss_vgg

        # Backward pass
        total_loss.backward()
        optimizer.step()

        # Статистика
        running_total += total_loss.item()
        running_l1 += loss_l1.item()
        running_vgg += loss_vgg.item()

        loop.set_postfix(loss=total_loss.item())

    # Сохраняем средние значения за эпоху
    history['total_loss'].append(running_total / len(train_loader))
    history['l1_loss'].append(running_l1 / len(train_loader))
    history['vgg_loss'].append(running_vgg / len(train_loader))

    # Визуализация прогресса
    if (epoch + 1) % 1 == 0:
        show_prediction(model, l_batch, ab_batch, epoch + 1)

    # Сохранение весов (чекпоинт)
    torch.save(model.state_dict(), 'resnet_unet_lab_latest.pth')

print("Обучение завершено!")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history['total_loss'], label='Total Loss', color='blue', linewidth=2)
plt.plot(history['l1_loss'], label='L1 Color Loss', linestyle='--', color='green')
plt.plot(history['vgg_loss'], label='VGG Perceptual Loss', linestyle='--', color='red')

plt.title('История обучения модели колоризации', fontsize=14)
plt.xlabel('Эпоха', fontsize=12)
plt.ylabel('Значение Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('loss_plot.png')
plt.show()

print(f"Минимальный лосс достигнут: {min(history['total_loss']):.4f}")